In [0]:
from pyspark.sql.functions import (
    sequence,
    explode,
    to_date,
    date_format,
    year,
    quarter,
    month,
    weekofyear,
    dayofmonth,
    dayofweek,
    when,
    col,
    lit,
    expr
)

DATE_TABLE = "workspace.default.dim_date"

df_date = (
    spark.range(1)
    .select(
        explode(
            sequence(
                to_date(lit("2026-01-01")),
                to_date(lit("2026-05-31")),
                expr("interval 1 day")
            )
        ).alias("date")
    )
)

In [0]:
df_date = (
    df_date
    .withColumn(
        "date_key",
        date_format("date", "yyyyMMdd").cast("integer")
    )
    .withColumn("year", year("date"))
    .withColumn("quarter", quarter("date"))
    .withColumn("month", month("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("week_of_year", weekofyear("date"))
    .withColumn("day_of_month", dayofmonth("date"))
    .withColumn("day_of_week", dayofweek("date"))
    .withColumn("day_name", date_format("date", "EEEE"))
    .withColumn(
        "is_weekend",
        when(dayofweek("date").isin([1, 7]), True)
        .otherwise(False)
    )
    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "day_name",
        "is_weekend"
    )
)

In [0]:
print("Rows:", df_date.count())
print("Columns:", len(df_date.columns))

display(
    df_date.orderBy("date").limit(10)
)

In [0]:
display(
    df_date
    .groupBy("is_weekend")
    .count()
)

In [0]:
display(
    df_date
    .groupBy("date_key")
    .count()
    .filter(col("count") > 1)
)

In [0]:
(
    df_date
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DATE_TABLE)
)

In [0]:
display(
    spark.table(DATE_TABLE)
    .orderBy("date")
)